# Отчёт: Отношение к банкам-маркетплейсам

> **Шаблон**: меняйте только **Блок 1** — пути к файлам и параметры.
> Остальные блоки не трогайте.


In [ ]:
# ╔══════════════════════════════╗
# ║  БЛОК 1 — НАСТРОЙКИ (менять)║
# ╚══════════════════════════════╝

# Путь к файлу с данными
INPUT_FILE  = r"C:\Users\Андрей\.gemini\antigravity\playground\Маркетплейсы\База итоговая.xlsx"

# Куда сохранить отчёт
OUTPUT_FILE = r"C:\Users\Андрей\.gemini\antigravity\playground\Маркетплейсы\Отчет_Маркетплейсы_ИТОГ.xlsx"

# Уровень значимости Z-теста
ALPHA = 0.05


In [ ]:
# ╔══════════════════════════════╗
# ║  БЛОК 2 — КОД (не менять)   ║
# ╚══════════════════════════════╝
# Установка: pip install openpyxl pandas scipy

"""
Генерация Excel-отчёта по банкам-маркетплейсам.
Используется из Jupyter Lab — запускайте ноутбук Marketplaces_Report.ipynb
"""
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings('ignore')

# ─── ЦВЕТА ─────────────────────────────────────────────────────────────────
FILL_DARK  = PatternFill("solid", fgColor="0070C0")   # синий заголовок
FILL_LIGHT = PatternFill("solid", fgColor="BDD7EE")   # голубой заголовок
FILL_GREEN = PatternFill("solid", fgColor="C3D69B")   # значимые ячейки

def bold(size=9, white=False):
    return Font(bold=True, size=size, name="Calibri", color="FFFFFF" if white else "000000")

def reg(size=9):
    return Font(bold=False, size=size, name="Calibri")

WRAP   = Alignment(wrap_text=True, vertical="center", horizontal="center")
LEFT   = Alignment(wrap_text=True, vertical="center", horizontal="left")
CENTER = Alignment(horizontal="center", vertical="center")

# ─── Z-ТЕСТ ────────────────────────────────────────────────────────────────
LETTERS = "abcdefghijklmnopqrstuvwxyz"

def z_test_sig(p1, n1, p2, n2, alpha=0.05):
    if n1 < 5 or n2 < 5 or (p1 == 0 and p2 == 0):
        return False
    p_pool = (p1 * n1 + p2 * n2) / (n1 + n2)
    if p_pool in (0, 1):
        return False
    se = (p_pool * (1 - p_pool) * (1/n1 + 1/n2)) ** 0.5
    if se == 0:
        return False
    from scipy.stats import norm
    z = (p1 - p2) / se
    return norm.cdf(z) > (1 - alpha / 2)

def apply_ztest(values, bases, compare_groups, alpha=0.05):
    """Применяет Z-тест внутри каждой группы сравнения."""
    n = len(values)
    winners = [[] for _ in range(n)]
    for grp in compare_groups:
        for pos_i, gi in enumerate(grp):
            if gi >= n: continue
            for pos_j, gj in enumerate(grp):
                if gi == gj or gj >= n: continue
                vi, vj = values[gi], values[gj]
                ni, nj = bases[gi],  bases[gj]
                if vi is not None and vj is not None and ni > 0 and nj > 0:
                    if z_test_sig(vi, ni, vj, nj, alpha):
                        winners[gi].append(LETTERS[pos_j])
    result = []
    for i, val in enumerate(values):
        if val is None:
            result.append(None)
        elif winners[i]:
            lets = "".join(sorted(set(winners[i])))
            result.append(f"{round(val*100)}% {lets}")
        else:
            result.append(round(val, 2))
    return result

# Группы сравнения (индексы столбцов 0-based, где 0=В целом)
COMPARE_GROUPS = [
    [1, 2, 3],       # WB(a), Ozon(b), Яндекс(c)
    [4, 5],          # WB: 3-28(a), WB: 29+(b)
    [6, 7],          # Ozon: 3-28(a), Ozon: 29+(b)
    [8, 9],          # Яндекс: 3-28(a), Яндекс: 29+(b)
    [10, 11],        # WB: 0-30(a), WB: 31-100(b)
    [12, 13],        # Ozon: 0-30(a), Ozon: 31-100(b)
    [14, 15],        # Яндекс: 0-30(a), Яндекс: 31-100(b)
    [16, 17],        # WB: <10k(a), WB: >10k(b)
    [18, 19],        # Ozon: <10k(a), Ozon: >10k(b)
    [20, 21],        # Яндекс: <10k(a), Яндекс: >10k(b)
]

# ─── МАППИНГ ОТВЕТОВ ───────────────────────────────────────────────────────
# Отображаемые строки → реальные ответы в данных для каждого банка

Q2_ANSWERS = {
    "display": [
        "Для совершения покупок со скидкой только на маркетплейсе",
        "Для совершения повседневных покупок вне маркетплейса",
        "В этом банке моя основная карта (повседневные покупки вне маркет.)",
        "Для открытия вклада/накопительного счёта",
        "Для хранения сбережений на вкладе/накопительном счёте",
        "Другое",
    ],
    "WB": [
        "Для совершения покупок со скидкой только на Wildberries",
        None,
        "В этом банке моя основная карта, с которой я совершаю повседневные покупки вне Wildberries",
        "Для открытия вклада/накопительного счета в этом банке",
        "Для хранения сбережений на вкладе/накопительном счете",
        "Другое",
    ],
    "Ozon": [
        "Для совершения покупок со скидкой только на Ozon",
        None,
        "В этом банке моя основная карта, с которой я совершаю повседневные покупки вне Ozon",
        "Для открытия вклада/накопительного счета в этом банке",
        "Для хранения сбережений на вкладе/накопительном счете",
        "Другое",
    ],
    "Яндекс": [
        "Для совершения покупок со скидкой только на Яндекс.Маркете и других сервисах Яндекса",
        None,
        "В этом банке моя основная карта, с которой я совершаю повседневные покупки вне Яндекс.Маркета и других сервисов Яндекса",
        "Для открытия вклада/накопительного счета в этом банке",
        "Для хранения сбережений на вкладе/накопительном счете",
        "Другое",
    ],
}

Q3_ANSWERS = {
    "display": [
        "Выгодный кешбэк/бонусы на покупки на маркетплейсе",
        "Выгодный кешбэк/бонусы на покупки вне маркетплейса",
        "Удобно, что финансы и покупки внутри одной системы",
        "Доступна мгновенная рассрочка на покупку внутри маркетплейса",
        "Привлекла рекламная акция",
        "Привлекли условия накопительного счёта",
        "Другое (укажите, что именно?)",
        "Затрудняюсь ответить",
    ],
    "WB": [
        "Выгодный кешбэк/бонусы на покупки на Wildberries",
        "Выгодный кешбэк/бонусы на покупки вне Wildberries",
        "Удобно, что финансы и покупки находятся внутри одной системы",
        "Доступна мгновенная рассрочка на покупку внутри Wildberries (Частями)",
        "Привлекла рекламная акция",
        None,
        "Другое",
        "Затрудняюсь ответить",
    ],
    "Ozon": [
        "Выгодный кешбэк/бонусы на покупки на Ozon",
        "Выгодный кешбэк/бонусы на покупки вне Ozon",
        "Удобно, что финансы и покупки находятся внутри одной системы",
        "Доступна мгновенная рассрочка на покупку внутри Ozon (Ozon Рассрочка)",
        "Привлекла рекламная акция",
        "Привлекли условия накопительного счета с начислением на ежедневный остаток",
        "Другое",
        "Затрудняюсь ответить",
    ],
    "Яндекс": [
        "Выгодный кешбэк/бонусы на покупки на Яндекс.Маркете и других сервисах Яндекса",
        "Выгодный кешбэк/бонусы на покупки вне Яндекс.Маркета и других сервисов Яндекса",
        "Удобно, что финансы и покупки находятся внутри одной системы",
        "Доступна мгновенная рассрочка на покупку внутри Яндекс.Маркета и других сервисов Яндекса (Сплит)",
        "Привлекла рекламная акция",
        "Привлекли условия накопительного счета с начислением на ежедневный остаток",
        "Другое",
        "Затрудняюсь ответить",
    ],
}

Q5_ANSWERS = {
    "display": [
        "Полностью соответствует",
        "Скорее соответствует",
        "Отчасти соответствует, отчасти нет",
        "Скорее не соответствует",
        "Полностью не соответствует",
        "Затрудняюсь ответить",
    ],
    "all": [  # одинаковые для всех банков
        "Полностью соответствует",
        "Скорее соответствует",
        "Отчасти соответствует, отчасти нет",
        "Скорее не соответствует",
        "Полностью не соответствует",
        "Затрудняюсь ответить",
    ],
}

Q6_ANSWERS = {
    "display": [
        "Кешбэк/бонусы на покупки на маркетплейсе ниже обещанного",
        "Кешбэк/бонусы на покупки вне маркетплейса ниже обещанного",
        "Неудобное приложение для совершения финансовых операций",
        "Отсутствие обновлений в приложении, финансовых продуктах",
        "«Фейковые», обманчивые скидки",
        "Некачественная техподдержка пользователей",
        "Неинтересные предложения по вкладам, накопительным счетам",
        "Нет нужных функций в приложении",
        "Другое (укажите, что именно?)",
        "Затрудняюсь ответить",
    ],
    "WB": [
        "Кешбэк/бонусы на покупки на Wildberries ниже обещанного",
        "Кешбэк/бонусы на покупки вне Wildberries ниже обещанного",
        "Неудобное приложение для совершения финансовых операций",
        "Отсутствие обновлений в приложении, финансовых продуктах",
        "«Фейковые», обманчивые скидки",
        "Некачественная техподдержка пользователей",
        "Неинтересные предложения по вкладам, накопительным счетам, кредитным картам и другим продуктам",
        "Нет нужных функций в приложении",
        "Другое",
        "Затрудняюсь ответить",
    ],
    "Ozon": [
        "Кешбэк/бонусы на покупки на Ozon ниже обещанного",
        "Кешбэк/бонусы на покупки вне Ozon ниже обещанного",
        "Неудобное приложение для совершения финансовых операций",
        "Отсутствие обновлений в приложении, финансовых продуктах",
        "«Фейковые», обманчивые скидки",
        "Некачественная техподдержка пользователей",
        "Неинтересные предложения по вкладам, накопительным счетам, кредитным картам и другим продуктам",
        "Нет нужных функций в приложении",
        "Другое",
        "Затрудняюсь ответить",
    ],
    "Яндекс": [
        "Кешбэк/бонусы на покупки на Яндекс.Маркете и других сервисах Яндекса ниже обещанного",
        "Кешбэк/бонусы на покупки вне Яндекс.Маркета и других сервисов Яндекса ниже обещанного",
        "Неудобное приложение для совершения финансовых операций",
        "Отсутствие обновлений в приложении, финансовых продуктах",
        "«Фейковые», обманчивые скидки",
        "Некачественная техподдержка пользователей",
        "Неинтересные предложения по вкладам, накопительным счетам, кредитным картам и другим продуктам",
        "Нет нужных функций в приложении",
        "Другое",
        "Затрудняюсь ответить",
    ],
}

Q7_ANSWERS = {
    "display": [
        "Банк-маркетплейс – удобный дополнительный функционал для покупок на маркетплейсе",
        "Я не хочу заводить регулярные платежи (ЖКХ, связь и т.д.) в банке-маркетплейсе",
        "Банк-маркетплейс не «дотягивает» до уровня традиционных банков",
        "Банк-маркетплейс выделяется среди других банков-маркетплейсов выгодными условиями",
        "Банк-маркетплейс обещает большой кешбэк, но не выполняет обещания",
        "Банк-маркетплейс не развивается как банк",
        "Продукты и условия банка-маркетплейса ухудшаются со временем",
        "Не согласен(-на) ни с одним из утверждений",
    ],
    "WB": [
        "WB Банк/ WB кошелек – это удобный дополнительный функционал для совершения покупок на Wildberries",
        "Я не хочу заводить регулярные платежи (ЖКХ, мобильная связь и т.д.) в WB Банк/ WB кошелек",
        "WB Банк/ WB кошелек не «дотягивает» до уровня традиционных банков из-за ограниченного количества финансовых продуктов",
        "WB Банк/ WB кошелек выделяется среди других банков-маркетплейсов выгодными условиями",
        "WB Банк/ WB кошелек обещает большой кешбэк, но не выполняет свои обещания",
        "WB Банк/ WB кошелек не развивается как банк",
        "Продукты и условия WB Банка/ WB кошелька ухудшаются со временем",
        "Не согласен(-на) ни с одним из утверждений",
    ],
    "Ozon": [
        "Ozon Банк – это удобный дополнительный функционал для совершения покупок на Ozon",
        "Я не хочу заводить регулярные платежи (ЖКХ, мобильная связь и т.д.) в Ozon банк",
        "Ozon Банк не «дотягивает» до уровня традиционных банков из-за ограниченного количества финансовых продуктов",
        "Ozon Банк выделяется среди других банков-маркетплейсов выгодными условиями на банковские продукты",
        "Ozon Банк обещает большой кешбэк, но не выполняет свои обещания",
        "Ozon Банк не развивается как банк",
        "Продукты и условия Ozon банк ухудшаются со временем",
        "Не согласен(-на) ни с одним из утверждений",
    ],
    "Яндекс": [
        "Яндекс.Банк – это удобный дополнительный функционал для совершения покупок в экосистеме Яндекс",
        "Я не хочу заводить регулярные платежи (ЖКХ, мобильная связь и т.д.) в Яндекс.Банке",
        "Яндекс.Банк не «дотягивает» до уровня традиционных банков из-за ограниченного количества финансовых продуктов",
        "Яндекс.Банк выделяется среди других банков-маркетплейсов выгодными условиями на банковские продукты",
        "Яндекс.Банк обещает большой кешбэк, но не выполняет свои обещания",
        "Яндекс.Банк не развивается как банк",
        "Продукты и условия Яндекс.Банка ухудшаются со временем",
        "Не согласен(-на) ни с одним из утверждений",
    ],
}

Q8_ANSWERS = {
    "display": [
        "Планирую пользоваться чаще",
        "Планирую пользоваться так же часто как и сейчас",
        "Планирую пользоваться реже или прекратить пользоваться",
    ],
    "all": [
        "Планирую пользоваться чаще",
        "Планирую пользоваться так же часто как и сейчас",
        "Планирую пользоваться реже или прекратить пользоваться",
    ],
}

Q9_ANSWERS = {
    "display": [
        "Выгодный кешбэк/бонусы на покупки на маркетплейсе",
        "Выгодный кешбэк/бонусы на покупки вне маркетплейса",
        "Удобно, что финансы и покупки внутри одной системы",
        "Доступна мгновенная рассрочка на покупку внутри маркетплейса",
        "Другое (укажите, что именно?)",
        "Затрудняюсь ответить",
    ],
    "WB": [
        "Выгодный кешбэк/бонусы на покупки на Wildberries",
        "Выгодный кешбэк/бонусы на покупки вне Wildberries",
        "Удобно, что финансы и покупки находятся внутри одной системы",
        "Доступна мгновенная рассрочка на покупку внутри Wildberries (Частями)",
        "Другое",
        None,
    ],
    "Ozon": [
        "Выгодный кешбэк/бонусы на покупки на Ozon",
        "Выгодный кешбэк/бонусы на покупки вне Ozon",
        "Удобно, что финансы и покупки находятся внутри одной системы",
        "Доступна мгновенная рассрочка на покупку внутри Ozon (Ozon Рассрочка)",
        "Другое",
        None,
    ],
    "Яндекс": [
        "Выгодный кешбэк/бонусы на покупки на Яндекс.Маркете и других сервисах Яндекса",
        "Выгодный кешбэк/бонусы на покупки вне Яндекс.Маркета и других сервисов Яндекса",
        "Удобно, что финансы и покупки находятся внутри одной системы",
        "Доступна мгновенная рассрочка на покупку внутри Яндекс.Маркета и других сервисов Яндекса (Сплит)",
        "Другое",
        None,
    ],
}

# ─── СТОЛБЦЫ В ДАННЫХ ──────────────────────────────────────────────────────
BANK_COLS = {
    "WB": {
        "q1": "0. Какими из банков ниже Вы пользуетесь?",
        "q2": "1. С какой целью вы сейчас пользуетесь WB Банком/ WB кошельком?",
        "q3": "2. Почему вы решили стать клиентом WB Банка/ WB кошелька?",
        "q5": "4. Насколько WB Банк/ WB кошелек соответствует ожиданиям, которые у вас были, когда вы стали клиентом банка?",
        "q6": "5. Почему WB Банк/ WB кошелек не вполне соответствует ожиданиям, которые были у вас, когда вы стали клиентом банка?",
        "q7": "6. Выберите утверждения о WB Банке/ WB кошельке, с которыми вы согласны",
        "q8": "7. Как вы планируете пользоваться услугами WB Банка/ WB кошелька для повседневных покупок в ближайшие 12 месяцев?",
        "q9": "8. Почему вы будете дальше пользоваться WB Банком/ WB кошельком?",
    },
    "Ozon": {
        "q1": "0. Какими из банков ниже Вы пользуетесь?",
        "q2": "1. С какой целью вы сейчас пользуетесь Ozon Банком?",
        "q3": "2. Почему вы решили стать клиентом Ozon Банка?",
        "q5": "4. Насколько Ozon Банк соответствует ожиданиям, которые у вас были, когда вы стали клиентом банка?",
        "q6": "5. Почему Ozon Банк не вполне соответствует ожиданиям, которые были у вас, когда вы стали клиентом банка?",
        "q7": "6. Выберите утверждения об Ozon Банке, с которыми вы согласны",
        "q8": "7. Как вы планируете пользоваться услугами Ozon Банка для повседневных покупок в ближайшие 12 месяцев?",
        "q9": "8. Почему вы будете дальше пользоваться Ozon Банком?",
    },
    "Яндекс": {
        "q1": "0. Какими из банков ниже Вы пользуетесь?",
        "q2": "1. С какой целью вы сейчас пользуетесь Яндекс.Банком?",
        "q3": "2. Почему вы решили стать клиентом Яндекс.Банка?",
        "q5": "4. Насколько Яндекс.Банк соответствует ожиданиям, которые у вас были, когда вы стали клиентом банка?",
        "q6": "5. Почему Яндекс.Банк не вполне соответствует ожиданиям, которые были у вас, когда вы стали клиентом банка?",
        "q7": "6. Выберите утверждения о Яндекс.Банке, с которыми вы согласны",
        "q8": "7. Как вы планируете пользоваться услугами Яндекс.Банка для повседневных покупок в ближайшие 12 месяцев?",
        "q9": "8. Почему вы будете дальше пользоваться Яндекс.Банком?",
    },
}

BANK_LABELS = {
    "Wildberries (Вайлдберриз Банк)": "WB",
    "Озон Банк (Ozon)": "Ozon",
    "Яндекс": "Яндекс",
}
Q1_BANK_ANSWERS = {
    "Ozon Банк": "Ozon Банк",
    "WB Банк/ WB кошелек": "WB Банк/ WB кошелек",
    "Яндекс.Банк": "Яндекс.Банк",
    "Никакими из перечисленных": None,
}

# ─── ЧТЕНИЕ ДАННЫХ ─────────────────────────────────────────────────────────
def load_data(filepath):
    df = pd.read_excel(filepath, sheet_name="База_ИТОГ")
    if "Анкета полная" in df.columns:
        df = df[df["Анкета полная"] == "Да"].copy()
    df["bank_key"] = df["bank_target"].map(BANK_LABELS)
    return df

def contains(series, value):
    """True если value входит в мультивыбор (разделитель |)."""
    if value is None:
        return pd.Series([False] * len(series), index=series.index)
    return series.astype(str).apply(
        lambda x: any(p.strip() == value for p in x.split("|"))
    )

# ─── СЕГМЕНТЫ ──────────────────────────────────────────────────────────────
def build_segs(df):
    bWB   = df["bank_key"] == "WB"
    bOzon = df["bank_key"] == "Ozon"
    bYa   = df["bank_key"] == "Яндекс"
    seg = df["Кол-во переводов (группы)"]
    sha = df["Доля переводов от поступлений (группы)"]
    sma = df["Сумма переводов по группам"]
    return [
        ("В целом",        None, None),
        ("WB",             bWB,  "WB"),
        ("Ozon",           bOzon,"Ozon"),
        ("Яндекс",         bYa,  "Яндекс"),
        ("WB_3-28",        bWB   & (seg == "3-28 переводов"),          "WB"),
        ("WB_29+",         bWB   & (seg == "29 переводов и более"),    "WB"),
        ("Ozon_3-28",      bOzon & (seg == "3-28 переводов"),          "Ozon"),
        ("Ozon_29+",       bOzon & (seg == "29 переводов и более"),    "Ozon"),
        ("Яндекс_3-28",    bYa   & (seg == "3-28 переводов"),          "Яндекс"),
        ("Яндекс_29+",     bYa   & (seg == "29 переводов и более"),    "Яндекс"),
        ("WB_0-30",        bWB   & (sha == "0-30%"),                   "WB"),
        ("WB_31-100",      bWB   & (sha == "31-100%"),                 "WB"),
        ("Ozon_0-30",      bOzon & (sha == "0-30%"),                   "Ozon"),
        ("Ozon_31-100",    bOzon & (sha == "31-100%"),                 "Ozon"),
        ("Яндекс_0-30",    bYa   & (sha == "0-30%"),                   "Яндекс"),
        ("Яндекс_31-100",  bYa   & (sha == "31-100%"),                 "Яндекс"),
        ("WB_lt10k",       bWB   & (sma == "до 10000 руб"),            "WB"),
        ("WB_gt10k",       bWB   & (sma == "больше 10 000 руб"),       "WB"),
        ("Ozon_lt10k",     bOzon & (sma == "до 10000 руб"),            "Ozon"),
        ("Ozon_gt10k",     bOzon & (sma == "больше 10 000 руб"),       "Ozon"),
        ("Яндекс_lt10k",   bYa   & (sma == "до 10000 руб"),            "Яндекс"),
        ("Яндекс_gt10k",   bYa   & (sma == "больше 10 000 руб"),       "Яндекс"),
    ]

# ─── РАСЧЁТ ОДНОЙ СТРОКИ ───────────────────────────────────────────────────
def calc_row(df, segs, qkey, answers_dict, answer_idx, alpha=0.05):
    """
    Считает одну строку таблицы для answer_idx-го ответа.
    answers_dict: {'display':[], 'WB':[], 'Ozon':[], 'Яндекс':[], 'all':[]}
    """
    vals, ns = [], []
    for seg_name, mask, bank_key in segs:
        sub = df if mask is None else df[mask]
        # Определяем ключ банка для ответа
        bk = bank_key  # None для "В целом"
        if bk is None:
            # В целом — смотрим по всем банкам через общий q1
            col = "0. Какими из банков ниже Вы пользуетесь?"
            if qkey != "q1":
                # Для остальных вопросов — считаем среднее по всем записям
                all_vals = []
                for bk2 in ["WB", "Ozon", "Яндекс"]:
                    ans_key = "all" if "all" in answers_dict else bk2
                    ans_val = answers_dict.get(ans_key, answers_dict.get("WB", []))[answer_idx]
                    col2 = BANK_COLS[bk2][qkey]
                    if col2 and col2 in df.columns:
                        bmask = df["bank_key"] == bk2
                        sub2 = df[bmask].dropna(subset=[col2])
                        if len(sub2):
                            all_vals.append(contains(sub2[col2], ans_val).sum() / len(sub2))
                if all_vals:
                    vals.append(np.mean(all_vals))
                    ns.append(len(df.dropna(subset=[BANK_COLS["WB"][qkey]] if BANK_COLS["WB"][qkey] in df.columns else [])))
                else:
                    vals.append(None); ns.append(0)
                continue
        # Получаем реальный ответ для данного банка
        ans_key = "all" if "all" in answers_dict else bk
        if ans_key not in answers_dict:
            vals.append(None); ns.append(0); continue
        ans_val = answers_dict[ans_key][answer_idx]
        col = BANK_COLS.get(bk, {}).get(qkey) if bk else None
        if qkey == "q1":
            # Q1 — смотрим в столбце q1 наличие банка
            q1_col = BANK_COLS["WB"]["q1"]
            sub2 = sub.dropna(subset=[q1_col]) if q1_col in df.columns else sub
            # Для Q1 ans_val — это название банка в ответе
            q1_ans_map = {
                "WB": "WB Банк/ WB кошелек",
                "Ozon": "Ozon Банк",
                "Яндекс": "Яндекс.Банк",
            }
            # Отображаемый ответ: Ozon Банк, WB Банк/ WB кошелек, Яндекс.Банк, Никакими
            display_ans = answers_dict["display"][answer_idx]
            # Маппинг отображаемого → значение в данных
            q1_val_map = {
                "Ozon Банк": "Ozon Банк",
                "WB Банк/ WB кошелек": "WB Банк/ WB кошелек",
                "Яндекс.Банк": "Яндекс.Банк",
                "Никакими из перечисленных": "Никакими из перечисленных",
            }
            lookup = q1_val_map.get(display_ans, display_ans)
            n = len(sub2)
            if n == 0: vals.append(None); ns.append(0); continue
            hits = contains(sub2[q1_col], lookup).sum()
            vals.append(hits / n); ns.append(n)
            continue
        if not col or col not in df.columns:
            vals.append(None); ns.append(0); continue
        sub2 = sub.dropna(subset=[col])
        n = len(sub2)
        if n == 0: vals.append(None); ns.append(0); continue
        if ans_val is None:
            vals.append(None); ns.append(n); continue
        hits = contains(sub2[col], ans_val).sum()
        vals.append(hits / n); ns.append(n)
    return vals, ns

def calc_table(df, segs, qkey, answers_dict, alpha=0.05):
    """Считает полную таблицу одного вопроса."""
    display = answers_dict["display"]
    rows = []
    all_vals_list = []
    all_ns_list   = []
    for i, disp in enumerate(display):
        vals, ns = calc_row(df, segs, qkey, answers_dict, i, alpha)
        all_vals_list.append(vals)
        all_ns_list.append(ns)
    # Вычисляем базы
    base_ns = all_ns_list[0] if all_ns_list else [0]*len(segs)
    # Для Q5 нужны не только строки, но и Top-2/Bottom-2
    labeled_rows = []
    for i, disp in enumerate(display):
        labeled = apply_ztest(all_vals_list[i], base_ns, COMPARE_GROUPS, alpha)
        labeled_rows.append([disp] + labeled)
    # Дополнительные строки для Q5
    if qkey == "q5":
        top2_indices = [0, 1]
        bot2_indices = [3, 4]
        top2_vals = [
            sum(v for v in [all_vals_list[ti][ci] for ti in top2_indices] if v is not None)
            if any(v is not None for v in [all_vals_list[ti][ci] for ti in top2_indices]) else None
            for ci in range(len(segs))
        ]
        bot2_vals = [
            sum(v for v in [all_vals_list[bi][ci] for bi in bot2_indices] if v is not None)
            if any(v is not None for v in [all_vals_list[bi][ci] for bi in bot2_indices]) else None
            for ci in range(len(segs))
        ]
        t2_labeled = apply_ztest(top2_vals, base_ns, COMPARE_GROUPS, alpha)
        b2_labeled = apply_ztest(bot2_vals, base_ns, COMPARE_GROUPS, alpha)
        labeled_rows.append(["Top-2 (полностью+скорее соответствует)"] + t2_labeled)
        labeled_rows.append(["Bottom-2 (полностью+скорее не соответствует)"] + b2_labeled)
    # База
    base_row = ["База"] + [n if n else "" for n in base_ns]
    labeled_rows.append(base_row)
    return labeled_rows

# ─── ЗАПИСЬ ЛИСТА ──────────────────────────────────────────────────────────
def write_title(ws, text, n_cols):
    c = ws.cell(row=1, column=1, value=text)
    c.fill = FILL_DARK; c.font = bold(11, white=True)
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols+2)
    ws.row_dimensions[1].height = 20

SEG_GROUPS = [
    ("В ЦЕЛОМ",                              [0]),
    ("Банк-маркетплейс",                     [1, 2, 3]),
    ("Количество переводов за 6 месяцев",    [4, 5, 6, 7, 8, 9]),
    ("Доля переводов от поступлений",        [10, 11, 12, 13, 14, 15]),
    ("Сумма переводов (группы)",             [16, 17, 18, 19, 20, 21]),
]

SEG_SUB_LABELS = {
    0: "В\nЦЕЛОМ",
    1: "WB Банк/\nWB кошелек\n(a)",
    2: "Ozon.Банк\n(b)",
    3: "ЯндексБанк\n(c)",
    4:  "WB:\n3-28 пер.\n(a)", 5:  "WB:\n29+\n(b)",
    6:  "Ozon:\n3-28 пер.\n(a)", 7:  "Ozon:\n29+\n(b)",
    8:  "Яндекс:\n3-28 пер.\n(a)", 9:  "Яндекс:\n29+\n(b)",
    10: "WB:\n0-30%\n(a)", 11: "WB:\n31-100%\n(b)",
    12: "Ozon:\n0-30%\n(a)", 13: "Ozon:\n31-100%\n(b)",
    14: "Яндекс:\n0-30%\n(a)", 15: "Яндекс:\n31-100%\n(b)",
    16: "WB:\nдо 10 тыс\n(a)", 17: "WB:\n>10 тыс\n(b)",
    18: "Ozon:\nдо 10 тыс\n(a)", 19: "Ozon:\n>10 тыс\n(b)",
    20: "Яндекс:\nдо 10 тыс\n(a)", 21: "Яндекс:\n>10 тыс\n(b)",
}

def write_q_block(ws, start_row, q_label, table_rows, n_segs):
    r = start_row
    # Название вопроса
    c = ws.cell(row=r, column=2, value=q_label)
    c.font = bold(9); c.alignment = LEFT
    r += 1
    # Строка групп
    for grp_name, idxs in SEG_GROUPS:
        valid = [i for i in idxs if i < n_segs]
        if not valid: continue
        col_start = valid[0] + 3
        col_end   = valid[-1] + 3
        c = ws.cell(row=r, column=col_start, value=grp_name)
        c.fill = FILL_LIGHT; c.font = bold(8); c.alignment = WRAP
        if col_end > col_start:
            ws.merge_cells(start_row=r, start_column=col_start, end_row=r, end_column=col_end)
    r += 1
    # Строка подзаголовков
    for ci in range(n_segs):
        lbl = SEG_SUB_LABELS.get(ci, "")
        c = ws.cell(row=r, column=ci+3, value=lbl)
        c.fill = FILL_LIGHT; c.font = bold(8); c.alignment = WRAP
        ws.row_dimensions[r].height = 45
    r += 1
    # Данные
    for dr in table_rows:
        label = dr[0]
        vals  = dr[1:]
        is_base = (label == "База")
        cell_b = ws.cell(row=r, column=2, value=label)
        cell_b.font = reg(9)
        cell_b.alignment = LEFT
        if not is_base:
            cell_b.fill = FILL_LIGHT

        for ci, val in enumerate(vals[:n_segs]):
            is_sig = isinstance(val, str) and "%" in val
            c = ws.cell(row=r, column=ci+3, value=val)
            c.alignment = CENTER
            c.font = bold(9) if is_sig else reg(9)
            c.fill = FILL_GREEN if is_sig else PatternFill(fill_type=None)
        r += 1
    r += 1  # пустая строка между блоками
    return r

def set_col_widths(ws, n_segs):
    ws.column_dimensions["A"].width = 3
    ws.column_dimensions["B"].width = 54
    ws.column_dimensions["C"].width = 17
    for i in range(1, n_segs):
        ws.column_dimensions[get_column_letter(i+3)].width = 11

def build_results(ws, df, alpha=0.05):
    segs = build_segs(df)
    n = len(segs)
    set_col_widths(ws, n)
    write_title(ws, "Сводные таблицы", n)

    # Q1 — особый: answers_dict должен иметь display + для каждого банка одинаковый ответ
    q1_dict = {
        "display": ["Ozon Банк", "WB Банк/ WB кошелек", "Яндекс.Банк", "Никакими из перечисленных"],
    }
    row = 3
    row = write_q_block(ws, row, "Q1. Какими из банков ниже Вы пользуетесь?",
                        calc_table(df, segs, "q1", q1_dict, alpha), n)
    row = write_q_block(ws, row, "Q2. С какой целью вы сейчас пользуетесь банком-маркетплейсом?",
                        calc_table(df, segs, "q2", Q2_ANSWERS, alpha), n)
    row = write_q_block(ws, row, "Q3. Почему вы решили стать клиентом банка-маркетплейса?",
                        calc_table(df, segs, "q3", Q3_ANSWERS, alpha), n)
    row = write_q_block(ws, row, "Q5. Насколько банк-маркетплейс соответствует ожиданиям?",
                        calc_table(df, segs, "q5", Q5_ANSWERS, alpha), n)
    row = write_q_block(ws, row, "Q6. Почему банк-маркетплейс не вполне соответствует ожиданиям?",
                        calc_table(df, segs, "q6", Q6_ANSWERS, alpha), n)
    row = write_q_block(ws, row, "Q7. Выберите утверждения о банке-маркетплейсе, с которыми вы согласны",
                        calc_table(df, segs, "q7", Q7_ANSWERS, alpha), n)
    row = write_q_block(ws, row, "Q8. Как вы планируете пользоваться услугами банка-маркетплейса?",
                        calc_table(df, segs, "q8", Q8_ANSWERS, alpha), n)
    row = write_q_block(ws, row, "Q9. Почему вы будете дальше пользоваться банком-маркетплейсом?",
                        calc_table(df, segs, "q9", Q9_ANSWERS, alpha), n)

def build_quotes(ws, df):
    ws.column_dimensions["A"].width = 20
    ws.column_dimensions["B"].width = 80
    write_title(ws, "Цитаты Другое", 2)
    row = 3
    for bk, full_name in [("WB", "WB Банк/ WB кошелек"), ("Ozon", "Ozon.Банк"), ("Яндекс", "ЯндексБанк")]:
        c = ws.cell(row=row, column=1, value=full_name)
        c.fill = FILL_LIGHT; c.font = bold(9)
        row += 1
        col = BANK_COLS[bk]["q3"]
        sub = df[df["bank_key"] == bk].dropna(subset=[col])
        standard = set(Q3_ANSWERS.get(bk, []))
        found = 0
        for _, r_data in sub.iterrows():
            parts = [p.strip() for p in str(r_data[col]).split("|")]
            custom = [p for p in parts if p and not any(
                p.startswith(s[:20]) for s in standard if s
            ) and p not in ("Другое", "Затрудняюсь ответить", "nan")]
            for txt in custom:
                ws.cell(row=row, column=1, value=full_name).font = reg(9)
                ws.cell(row=row, column=2, value=txt).font = reg(9)
                row += 1; found += 1
        # Также "Другое" open-ended если есть
        col9 = BANK_COLS[bk]["q9"]
        if col9 in df.columns:
            c_txt_col = None
            # Ответ "Другое" по Q9
            for _, r_data in df[df["bank_key"] == bk].iterrows():
                v = str(r_data.get(col9, ""))
                if "Другое" in v:
                    # Ищем открытый текст в соседних столбцах (нет — пропускаем)
                    pass
        if found == 0:
            ws.cell(row=row, column=2, value="—").font = reg(9)
            row += 1
        row += 1

def build_legend(ws):
    ws.column_dimensions["B"].width = 70
    c = ws.cell(row=2, column=2, value="Условные обозначения: как читать таблицы")
    c.font = bold(11); c.alignment = LEFT
    items = [
        (4, "Значения в таблицах — доли (0.75 = 75%). Значения < 0.01 могут отображаться как 0."),
        (5, "Зелёный фон + буква (напр. 84% ac) — значение значимо выше столбцов a и c (Z-тест, p<0.05)."),
        (6, "WB Банк/ WB кошелек (a), Ozon.Банк (b), ЯндексБанк (c) — буквенная кодировка столбцов."),
        (7, "В каждой секции сегментов (переводы, доля, сумма) сравнение идёт внутри одного банка: a vs b."),
        (8, "База — количество анкет, использованных для расчёта данного вопроса."),
        (9, "Анкета полная = Да — учитываются только полностью заполненные анкеты."),
    ]
    for r, txt in items:
        ws.cell(row=r, column=2, value=txt).font = reg(9)

# ─── ГЛАВНАЯ ФУНКЦИЯ ───────────────────────────────────────────────────────
def generate_report(input_path, output_path, alpha=0.05):
    print("Загружаем данные...")
    df = load_data(input_path)
    print(f"Строк после фильтра: {len(df)}")

    wb = Workbook()
    ws1 = wb.active; ws1.title = "Результаты"
    print("Строим лист Результаты...")
    build_results(ws1, df, alpha)

    ws2 = wb.create_sheet("Цитаты Другое")
    print("Строим лист Цитаты Другое...")
    build_quotes(ws2, df)

    ws3 = wb.create_sheet("Усл. обозначения")
    build_legend(ws3)

    wb.save(output_path)
    print(f"✅ Файл сохранён: {output_path}")
    return output_path


In [ ]:
# ╔══════════════════════════════╗
# ║  БЛОК 3 — ЗАПУСК (не менять)║
# ╚══════════════════════════════╝
result = generate_report(INPUT_FILE, OUTPUT_FILE)
print(f"\n✅ Готово! Файл: {result}")
